In [1]:
!pip install -q transformers datasets peft accelerate bitsandbytes
!pip install -q qwen-vl-utils torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 45.9 MB/s eta 0:00:00


In [2]:
import os
from google.colab import drive

# Make sure you sign into the account that OWNS the GSV_Math_Model_Cache folder!
drive.mount('/content/drive')

# Point directly to the checkpoint you just finished training
ADAPTER_PATH = "/content/drive/MyDrive/GSV_Math_Model_Cache/qwen25vl_math_expert_finetuned/checkpoint-2000"
RESULTS_FILE = "/content/drive/MyDrive/GSV_Math_Model_Cache/gsv_math_results/qwen25vl7b_finetuned_results.json"

os.makedirs(os.path.dirname(RESULTS_FILE), exist_ok=True)
print(f"Adapter Path: {ADAPTER_PATH}")
print(f"Results File: {RESULTS_FILE}")

Mounted at /content/drive
Adapter Path: /content/drive/MyDrive/GSV_Math_Model_Cache/qwen25vl_math_expert_finetuned/checkpoint-2000
Results File: /content/drive/MyDrive/GSV_Math_Model_Cache/gsv_math_results/qwen25vl7b_finetuned_results.json


In [3]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import PeftModel

MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

print("1. Loading Base Model in 4-bit...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

print(f"2. Applying Your Trained LoRA Adapter...")
# This merges your trained weights with the base model!
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

processor = AutoProcessor.from_pretrained(MODEL_ID)
print(" Fine-tuned Model Ready for Testing!")

1. Loading Base Model in 4-bit...


config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/57.6k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

 Fine-tuned Model Ready for Testing!


In [4]:
import re

FINAL_ANSWER_PATTERNS = [
    r'\\boxed\{([^}]*)\}',
    r'[Ff]inal\s*[Aa]nswer\s*[:\-]?\s*(.{1,80})',
    r'[Tt]herefore[,\\s]+(?:the\s+)?(?:answer|value|result)\s+is\s*[:\-]?\s*(.{1,80})',
    r'[Tt]he\s+answer\s+is\s*[:\-]?\s*(.{1,80})',
    r'[Ss]o\s+the\s+answer\s+is\s*[:\-]?\s*(.{1,80})',
    r'=\s*(\S+)\s*$',
]

def extract_final_answer_region(raw_text, tail_chars=300):
    for pattern in FINAL_ANSWER_PATTERNS:
        matches = list(re.finditer(pattern, raw_text, re.IGNORECASE | re.DOTALL))
        if matches: return matches[-1].group(1).strip()
    return raw_text[-tail_chars:] if len(raw_text) > tail_chars else raw_text

def clean_free_form(text):
    if not isinstance(text, str): return str(text)
    text = text.strip().lower()
    for prefix in ["the answer is", "therefore, the answer is", "so the answer is", "the value is", "answer is", "value is", "equals", "it is", "the final answer is", "final answer:", "answer:"]:
        if text.startswith(prefix):
            text = text[len(prefix):].strip()
    match = re.match(r'^[a-zA-Z\s]+=\s*(.*)$', text)
    if match: text = match.group(1).strip()
    return text.rstrip('.!?*, ')

def get_most_similar(extraction, choices):
    if not choices: return extraction
    distances = [-len(set(extraction.lower()) & set(choice.lower())) for choice in choices]
    return choices[distances.index(min(distances))]

def normalize_extracted_answer(extraction, choices, question_type, answer_type):
    extraction = str(extraction).strip() if extraction else ""
    extraction = extract_final_answer_region(extraction)

    if question_type == 'multi_choice':
        letter = re.findall(r'\(([a-zA-Z])\)', extraction)
        extraction = letter[0].upper() if letter else extraction
        options = [chr(ord('A') + i) for i in range(len(choices))]
        if extraction in options:
            extraction = choices[options.index(extraction)]
        else:
            extraction = get_most_similar(clean_free_form(extraction), choices)
    else:
        cleaned = clean_free_form(extraction)
        if answer_type in ['integer', 'float']:
            numbers = re.findall(r'-?\d+\.?\d*', cleaned)
            extraction = numbers[-1] if numbers else cleaned
        else:
            extraction = cleaned
    return extraction

def is_correct(pred, gt, answer_type):
    if str(pred).lower().strip() == str(gt).lower().strip(): return 1
    if answer_type in ['integer', 'float']:
        try:
            if abs(float(pred) - float(gt)) < 1e-5: return 1
        except: pass
    return 0

In [6]:
import json
import time
import gc
from tqdm.auto import tqdm
from datasets import load_dataset
from qwen_vl_utils import process_vision_info
import torch

print("Loading MathVista testmini...")
dataset = load_dataset("AI4Math/MathVista", split="testmini")

# Resume from where we left off if Colab crashes
results = []
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE, "r") as f:
        results = json.load(f)
completed_pids = {res["pid"] for res in results}
print(f"Resuming run: {len(completed_pids)}/{len(dataset)} samples already done.")

remaining_samples = [s for s in dataset if s["pid"] not in completed_pids]

print(f"Starting evaluation on {len(remaining_samples)} samples...")
for sample in tqdm(remaining_samples):
    pid = sample["pid"]
    question = sample["query"]
    image = sample["decoded_image"]
    choices = sample.get("choices", [])
    q_type = sample["question_type"]
    a_type = sample["answer_type"]
    gt = sample["answer"]

    messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": image,
                    "max_pixels": 313600  # <--- CRITICAL FIX: Stops the 33GB VRAM explosion!
                },
                {"type": "text", "text": question}
            ]
        }
    ]

    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)

    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt"
    ).to(model.device)

    # Generate the answer (Limit to 1024 tokens so it doesn't ramble)
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=1024)
        generated_ids_trimmed = [
            out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
        ]
        output_text = processor.batch_decode(
            generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
        )[0]

    parsed_ans = normalize_extracted_answer(output_text, choices, q_type, a_type)
    correct = is_correct(parsed_ans, gt, a_type)

    results.append({
        "pid": pid,
        "question": question,
        "raw_response": output_text,
        "parsed_response": parsed_ans,
        "ground_truth": gt,
        "question_type": q_type,
        "answer_type": a_type,
        "correct": correct
    })

    # Save every 10 questions safely
    if len(results) % 10 == 0:
        with open(RESULTS_FILE, "w") as f:
            json.dump(results, f, indent=4)
        time.sleep(2)  # Give Drive time to sync

    # --- MEMORY CLEANUP: Prevents slow memory leaks! ---
    del inputs, generated_ids, generated_ids_trimmed, messages, text, image_inputs, video_inputs
    gc.collect()
    torch.cuda.empty_cache()

# Final Save
with open(RESULTS_FILE, "w") as f:
    json.dump(results, f, indent=4)
print(" Evaluation Complete!")

Loading MathVista testmini...
Resuming run: 20/1000 samples already done.
Starting evaluation on 980 samples...


  0%|          | 0/980 [00:00<?, ?it/s]

 Evaluation Complete!


In [8]:
import json
from datasets import load_dataset

# Load the testmini dataset to get the categories
print("Loading MathVista dataset for categories...")
dataset = load_dataset("AI4Math/MathVista", split="testmini")

# Map each question ID to its skills
pid_to_skills = {s["pid"]: s["metadata"]["skills"] for s in dataset}

# Load your results
with open(RESULTS_FILE, "r") as f:
    results = json.load(f)

skill_to_category = {
    "geometry reasoning": "geometry",
    "arithmetic reasoning": "arithmetic",
    "algebraic reasoning": "algebra",
    "logical reasoning": "logic",
    "numeric commonsense": "numeric",
    "scientific reasoning": "scientific",
    "statistical reasoning": "statistical",
}

categories = ["all", "geometry", "arithmetic", "algebra", "logic", "numeric", "scientific", "statistical"]
metrics = {cat: {"correct": 0, "total": 0} for cat in categories}

for res in results:
    pid = res["pid"]
    correct = res["correct"]

    metrics["all"]["correct"] += correct
    metrics["all"]["total"] += 1

    # Get the skills for this specific question
    skills = pid_to_skills.get(pid, [])

    for skill in skills:
        cat = skill_to_category.get(skill)
        if cat in metrics:
            metrics[cat]["correct"] += correct
            metrics[cat]["total"] += 1

print("\n" + "="*55)
print(f"{'Qwen 2.5 Fine-Tuned (Real Image) Breakdown':^55}")
print("="*55)
print(f"{'Category':<20} | {'Correct':<10} | {'Total':<10} | {'Accuracy':<10}")
print("-" * 55)

for cat in categories:
    correct = metrics[cat]["correct"]
    total = metrics[cat]["total"]
    acc = (correct / total * 100) if total > 0 else 0.0
    print(f"{cat.capitalize():<20} | {correct:<10} | {total:<10} | {acc:.2f}%")

print("="*55)

Loading MathVista dataset for categories...

      Qwen 2.5 Fine-Tuned (Real Image) Breakdown       
Category             | Correct    | Total      | Accuracy  
-------------------------------------------------------
All                  | 683        | 1000       | 68.30%
Geometry             | 178        | 239        | 74.48%
Arithmetic           | 219        | 353        | 62.04%
Algebra              | 200        | 281        | 71.17%
Logic                | 6          | 37         | 16.22%
Numeric              | 64         | 144        | 44.44%
Scientific           | 81         | 122        | 66.39%
Statistical          | 226        | 301        | 75.08%
